# Ocean Box Transient CO2 Uptake
> Andrew McGallian

Closed air-ocean column model: how does an atmospheric CO2 spike equilibrate into the ocean, with and without CaCO3 compensation?

## CO2 System Solver (shared functions)

In [ ]:
import matplotlib.pyplot as plt

K1      = 10**(-6.5)
K2      = 10**(-10.8)
K_henry = 10**(1.5) / 1000   # mol/L/atm
K_water = 10**(-14)

def _alpha0(h): return h**2   / (h**2 + K1*h + K1*K2)
def _alpha1(h): return K1*h   / (h**2 + K1*h + K1*K2)
def _alpha2(h): return K1*K2  / (h**2 + K1*h + K1*K2)
def _alk_from_dic(DIC_L, h): return DIC_L*(_alpha1(h) + 2*_alpha2(h)) + K_water/h - h

def solve_co2_system(alk_m3, DIC_m3, iterations=100):
    """Solve CO2 system given alk and DIC in mol/m³. Returns (CO2_m3, CO3_m3, pCO2_ppm, pH)."""
    alk_L, DIC_L = alk_m3/1000, DIC_m3/1000
    pH_lo, pH_hi = 1.0, 14.0
    for _ in range(iterations):
        pH_mid = (pH_lo + pH_hi) / 2
        h = 10**(-pH_mid)
        if _alk_from_dic(DIC_L, h) < alk_L: pH_lo = pH_mid
        else:                                pH_hi = pH_mid
    pH = (pH_lo + pH_hi) / 2
    h  = 10**(-pH)
    CO2_L = DIC_L * _alpha0(h)
    CO3_L = DIC_L * _alpha2(h)
    return CO2_L*1000, CO3_L*1000, (CO2_L/K_henry)*1e6, pH

def co2_from_ppm(pCO2_ppm):
    """Dissolved CO2 concentration (mol/m³) for a given atmospheric pCO2 (ppm)."""
    return K_henry * (pCO2_ppm/1e6) * 1000

CO2, CO3, pCO2, pH = solve_co2_system(2.0, 2.0)
print(f"pH={pH:.2f}, pCO2={pCO2:.1f} ppm  (expected: pH≈8.59, pCO2≈507 ppm)")

## Ocean Box Model

In [ ]:
H_ATM   = 8000      # m — atmosphere column height
H_OCEAN = 4000      # m — ocean depth
K_GAS   = 3 * 365   # m/yr (3 m/day canonical gas exchange)
K_CACO3 = 10        # m/yr — CaCO3 dissolution rate constant

def ocean_box_model(co2_spike_ppm, caco3_enabled, dt=0.5, duration=1000):
    """
    Simulate transient CO2 uptake in a closed air-ocean box.

    co2_spike_ppm : atmospheric CO2 spike at t=0 (ppm)
    caco3_enabled : whether CaCO3 dissolution compensates CO3²⁻ drawdown
    """
    # Moles of air per m² column at STP (1 mol = 22.4 L = 0.0224 m³)
    moles_air   = H_ATM / 0.0224
    ppm_per_mol = 1e6 / moles_air

    alk0, DIC0 = 2.0, 2.0
    CO2_0, CO3_0, pCO2_0, _ = solve_co2_system(alk0, DIC0)

    atm_pCO2  = pCO2_0 + co2_spike_ppm
    atm_C_inv = atm_pCO2 / ppm_per_mol        # mol/m²
    alk_inv   = alk0 * H_OCEAN
    DIC_inv   = DIC0 * H_OCEAN

    # Seed output lists with initial values
    time_points    = [0.0]
    alk_conc       = [alk0];    DIC_conc    = [DIC0]
    CO2_conc       = [CO2_0];   CO3_conc    = [CO3_0]
    pCO2_ocean     = [pCO2_0]
    alk_inv_list   = [alk_inv]; DIC_inv_list = [DIC_inv]
    atm_pCO2_list  = [atm_pCO2]
    atm_C_inv_list = [atm_C_inv]
    atm_CO2_eq_list = [co2_from_ppm(atm_pCO2)]
    co2_flux_list   = [0.0]
    caco3_flux_list = [0.0]

    t = 0.0
    for _ in range(int(duration/dt)):
        t += dt
        time_points.append(t)

        alk_c = alk_inv / H_OCEAN
        DIC_c = DIC_inv / H_OCEAN
        CO2_c, CO3_c, pCO2_c, _ = solve_co2_system(alk_c, DIC_c)
        pCO2_ocean.append(pCO2_c)
        CO3_conc.append(CO3_c)
        CO2_conc.append(CO2_c)

        atm_CO2_eq = co2_from_ppm(atm_pCO2)
        atm_CO2_eq_list.append(atm_CO2_eq)

        co2_flux = K_GAS * (atm_CO2_eq - CO2_c)   # mol/m²/yr
        atm_C_inv -= co2_flux * dt
        atm_pCO2   = atm_C_inv * ppm_per_mol
        atm_pCO2_list.append(atm_pCO2)
        atm_C_inv_list.append(atm_C_inv)

        caco3_flux = K_CACO3 * (CO3_0 - CO3_c) if caco3_enabled else 0.0

        DIC_inv += (co2_flux + caco3_flux) * dt
        alk_inv += 2 * caco3_flux * dt

        alk_conc.append(alk_inv / H_OCEAN)
        DIC_conc.append(DIC_inv / H_OCEAN)
        alk_inv_list.append(alk_inv)
        DIC_inv_list.append(DIC_inv)
        co2_flux_list.append(co2_flux)
        caco3_flux_list.append(caco3_flux)

    return [time_points, alk_conc, DIC_conc, CO2_conc, CO3_conc,
            pCO2_ocean, alk_inv_list, DIC_inv_list,
            atm_pCO2_list, atm_C_inv_list, atm_CO2_eq_list,
            co2_flux_list, caco3_flux_list]

## Run experiments

In [ ]:
no_caco3   = ocean_box_model(co2_spike_ppm=200, caco3_enabled=False)
with_caco3 = ocean_box_model(co2_spike_ppm=200, caco3_enabled=True)

## Plots

In [ ]:
t_n  = no_caco3[0];   t_c  = with_caco3[0]
alk_n, DIC_n = no_caco3[1],  no_caco3[2]
alk_c, DIC_c = with_caco3[1], with_caco3[2]
CO3_n = no_caco3[4];  CO3_c = with_caco3[4]
atm_n = no_caco3[8];  atm_c = with_caco3[8]
pCO2_oc_n = no_caco3[5];  pCO2_oc_c = with_caco3[5]
flux_n  = no_caco3[11];  flux_c   = with_caco3[11]
caco3_c = with_caco3[12]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# pCO2
axes[0,0].plot(t_n, atm_n,     label='Atm (no CaCO3)')
axes[0,0].plot(t_n, pCO2_oc_n, label='Ocean eq (no CaCO3)', ls='--')
axes[0,0].plot(t_c, atm_c,     label='Atm (with CaCO3)')
axes[0,0].plot(t_c, pCO2_oc_c, label='Ocean eq (with CaCO3)', ls='--')
axes[0,0].set(xlabel='Time (yr)', ylabel='pCO2 (ppm)', title='Atmospheric & Ocean pCO2')
axes[0,0].legend(fontsize=7)

# CO32-
axes[0,1].plot(t_n, CO3_n, label='No CaCO3')
axes[0,1].plot(t_c, CO3_c, label='With CaCO3')
axes[0,1].set(xlabel='Time (yr)', ylabel='CO3²⁻ (mol/m³)', title='Ocean CO3²⁻')
axes[0,1].legend()

# Alk & DIC
axes[1,0].plot(t_n, alk_n, label='Alk (no CaCO3)')
axes[1,0].plot(t_n, DIC_n, label='DIC (no CaCO3)', ls='--')
axes[1,0].plot(t_c, alk_c, label='Alk (with CaCO3)')
axes[1,0].plot(t_c, DIC_c, label='DIC (with CaCO3)', ls='--')
axes[1,0].set(xlabel='Time (yr)', ylabel='mol/m³', title='Alkalinity & Total CO2')
axes[1,0].legend(fontsize=7)

# Fluxes
axes[1,1].plot(t_n, flux_n,   label='CO2 flux (no CaCO3)')
axes[1,1].plot(t_c, flux_c,   label='CO2 flux (with CaCO3)')
axes[1,1].plot(t_c, caco3_c,  label='CaCO3 flux', ls='--')
axes[1,1].set(xlabel='Time (yr)', ylabel='mol/m²/yr', title='CO2 & CaCO3 Fluxes')
axes[1,1].legend(fontsize=7)

for ax in axes.flatten():
    ax.grid(alpha=0.3)

plt.suptitle('Ocean Box CO2 Uptake: +200 ppm Spike', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()